# PRACTICE 1: CLASSIFYING SPAM EMAILS
## PART 1: KHÁM PHÁ DỮ LIỆU & LÀM SẠCH VĂN BẢN (DATA CLEANING & EDA)

```text
[CEAS_08.csv] ──> Mục 1 (EDA Nền tảng: Cấu trúc, Chất lượng & Đơn biến)
              ──> Mục 2 (EDA Chuyên sâu: Đa biến, Ngoại lai & Đúc kết định hướng)
              ──> Mục 3 (Làm sạch dữ liệu: Khử trùng, điền missing, clean_text) ──> [clean_emails.csv]
```


---
## 0. GIỚI THIỆU BỘ DỮ LIỆU THỰC NGHIỆM: CEAS_08
Dự án sử dụng bộ dữ liệu chuẩn mực **CEAS_08** (*Collaboration: Electronic Attack & Security Suite 2008*) — tập dữ liệu benchmark kinh điển trong các nghiên cứu quốc tế về phân loại thư rác và phát hiện tấn công lừa đảo qua email:

- **Quy mô tập dữ liệu:** Gồm **39.154 email** thực tế với kích thước tệp thô xấp xỉ 65 MB.
- **Phân phối biến mục tiêu (Ground Truth):**
  - **Spam (Thư rác - Nhãn 1):** 21.854 mẫu (**55.8%**)
  - **Ham (Thư hợp lệ - Nhãn 0):** 17.300 mẫu (**44.2%**)
  - *Đánh giá cân bằng:* Dữ liệu phân bố tương đối đồng đều giữa 2 lớp, không xảy ra hiện tượng mất cân bằng trầm trọng (Class Imbalance).

### 📋 Bảng Từ điển Dữ liệu (Data Dictionary):
| Tên thuộc tính | Kiểu dữ liệu | Vai trò nghiệp vụ | Ý nghĩa kỹ thuật |
| :--- | :---: | :---: | :--- |
| `sender` | Chuỗi (`object`) | Đặc trưng nguồn | Địa chỉ email của người gửi (dùng để bóc tách tên miền `sender_domain`) |
| `receiver` | Chuỗi (`object`) | Đặc trưng đích | Địa chỉ email của người nhận |
| `date` | Chuỗi (`object`) | Thời gian gửi | Mốc thời gian gửi thư trong siêu dữ liệu email |
| `subject` | Chuỗi (`object`) | Đặc trưng văn bản | Tiêu đề email (chứa 28 giá trị khuyết thiếu cần điền chuỗi rỗng) |
| `body` | Chuỗi (`object`) | Đặc trưng văn bản | Nội dung chi tiết của bức email |
| `urls` | Số nguyên (`int64`) | Đặc trưng số | Cờ đánh dấu sự hiện diện của liên kết (0: Không chứa URL, 1+: Có chứa URL) |
| `label` | Số nguyên (`int64`) | Biến mục tiêu (Target) | Nhãn phân loại nhị phân (0: Thư hợp lệ - Ham, 1: Thư rác - Spam) |

---
## Mục 1: Khám phá dữ liệu tổng quan — Cấu trúc, Chất lượng & Đơn biến (Overview EDA)


### 1.1 Khai báo Thư viện & Cấu hình Thực nghiệm
Để đảm bảo tính nhất quán và khả năng tái lập kết quả (**reproducibility**), ta tiến hành nạp các thư viện phân tích dữ liệu cốt lõi (`pandas`, `numpy`, `matplotlib`, `seaborn`) và cố định hạt giống ngẫu nhiên `RANDOM_SEED = 42`. Đồng thời, cấu hình hiển thị đầy đủ các cột và font chữ tiếng Việt để quá trình trực quan hóa thông suốt.

In [ ]:
# Khai báo thư viện & Thiết lập môi trường
import os
import sys
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')

DATA_RAW_PATH = '../data/raw/CEAS_08.csv'
print(f'✅ Môi trường sẵn sàng với Python {sys.version.split()[0]} | RANDOM_SEED = {RANDOM_SEED}')

**Nhận xét:**
- Môi trường làm việc đã được cấu hình thành công với chế độ mã hóa ký tự `utf-8`.
- Việc cố định seed giúp toàn bộ các phép chia mẫu hay lấy mẫu ngẫu nhiên ở các bước sau cho ra kết quả đồng nhất trên máy của tất cả thành viên trong nhóm.

### 1.2 Khảo sát Cấu trúc & Đánh giá Sơ bộ Chất lượng Dữ liệu
Bước đầu tiên trong mọi dự án khoa học dữ liệu là nắm bắt "hình hài" của tập dữ liệu thô. Ta cần trả lời 3 câu hỏi cơ bản:
1. Bộ dữ liệu có bao nhiêu dòng, bao nhiêu cột và kiểu dữ liệu từng cột là gì?
2. Có thuộc tính nào bị khuyết thiếu dữ liệu (**missing values**) hay không?
3. Tỷ lệ các bản ghi trùng lặp (**duplicates**) là bao nhiêu?

In [ ]:
# Khảo sát cấu trúc & Đánh giá chất lượng dữ liệu
df = pd.read_csv("../data/raw/CEAS_08.csv")
print(f"📊 Kích thước dữ liệu gốc: {df.shape[0]:,} dòng, {df.shape[1]} cột")
print("-" * 60)
print("Thông tin kiểu dữ liệu và bộ nhớ:")
df.info()

print("\nNăm dòng đầu tiên của dữ liệu:")
display(df.head(5))

print("\nThống kê mô tả các biến số:")
display(df.describe())

# Đánh giá giá trị khuyết thiếu (Missing values)
missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df)) * 100
missing_summary = pd.DataFrame({
    'Số lượng thiếu': missing_count,
    'Tỷ lệ thiếu (%)': missing_percent.round(4)
})
print("\nThống kê giá trị khuyết thiếu trên các cột:")
display(missing_summary[missing_summary['Số lượng thiếu'] > 0])

# Đánh giá bản ghi trùng lặp (Duplicate records)
duplicate_count = df.duplicated().sum()
print(f"\n• Số dòng trùng lặp hoàn toàn: {duplicate_count:,} dòng ({(duplicate_count/len(df))*100:.2f}%)")


**Kết luận từ kết quả khảo sát:**
- **Quy mô:** Bộ dữ liệu `CEAS_08.csv` gồm **39.154 dòng** và **6 cột** (`sender`, `receiver`, `date`, `subject`, `body`, `label`, `urls`).
- **Giá trị khuyết thiếu:** Cột `subject` (tiêu đề email) có **28 dòng bị thiếu (`NaN`)**, trong khi các cột khác đều đầy đủ 100%. Mặc dù tỷ lệ thiếu rất nhỏ (< 0.1%), ta vẫn bắt buộc phải xử lý điền chuỗi rỗng `''` để tránh gây lỗi `AttributeError` khi tiền xử lý văn bản ở Mục 3.
- **Dữ liệu trùng lặp:** Không phát hiện dòng dữ liệu nào bị trùng lặp hoàn toàn trên toàn bộ các trường (0 dòng trùng). Dữ liệu thô giữ được độ toàn vẹn ban đầu rất tốt.

### 1.3 Phân tích Đơn biến (Univariate Analysis)
Trước khi đi sâu vào mối quan hệ giữa các biến, ta khảo sát độc lập từng thuộc tính quan trọng đại diện cho 4 nhóm kiểu dữ liệu khác nhau:
- **Biến mục tiêu (`label`):** Kiểm tra xem bài toán có bị mất cân bằng dữ liệu (**Class Imbalance**) hay không.
- **Biến số nhị phân (`urls`):** Tần suất email có chứa đường dẫn URL.
- **Biến văn bản (`subject_len`):** Phân phối độ dài tiêu đề email.
- **Biến danh mục (`sender_domain`):** Thống kê các tên miền người gửi phổ biến nhất.

In [ ]:
# Phân tích đơn biến 4 thuộc tính
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Biến mục tiêu: label (0: Ham, 1: Spam)
label_counts = df['label'].value_counts()
sns.countplot(data=df, x='label', palette=['#3498db', '#e74c3c'], ax=axes[0, 0])
axes[0, 0].set_title(f"1. Biến mục tiêu: Ham ({label_counts.get(0, 0):,}) vs Spam ({label_counts.get(1, 0):,})")
axes[0, 0].set_xlabel('Nhãn (0: Ham / Hợp lệ, 1: Spam / Thư rác)')
axes[0, 0].set_ylabel('Số lượng email')
for p in axes[0, 0].patches:
    axes[0, 0].annotate(f"{int(p.get_height()):,} ({p.get_height()/len(df)*100:.1f}%)",
                         (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                         ha='center', va='center', color='white', fontweight='bold', fontsize=11)

# Biến số độc lập: urls (Có/Không link URL)
sns.countplot(data=df, x='urls', palette='viridis', ax=axes[0, 1])
axes[0, 1].set_title('2. Biến số URLs: Tỷ lệ email có chứa liên kết')
axes[0, 1].set_xlabel('Số lượng liên kết URL (0: Không, 1+: Có)')
axes[0, 1].set_ylabel('Số lượng email')

# Biến văn bản: subject length (Độ dài tiêu đề)
df['subject_len'] = df['subject'].fillna('').astype(str).apply(len)
sns.histplot(df['subject_len'], bins=50, kde=True, color='#2ecc71', ax=axes[1, 0])
axes[1, 0].set_title('3. Phân phối độ dài tiêu đề (Subject Length)')
axes[1, 0].set_xlabel('Số ký tự trong tiêu đề')
axes[1, 0].set_ylabel('Mật độ / Số lượng')
axes[1, 0].set_xlim(0, 200)

# Biến danh mục: sender domain (Tên miền người gửi)
df['sender_domain'] = df['sender'].astype(str).apply(lambda x: x.split('@')[-1].lower() if '@' in x else 'unknown')
top_domains = df['sender_domain'].value_counts().head(10)
top_domains.plot(kind='bar', color='#f39c12', ax=axes[1, 1])
axes[1, 1].set_title('4. Top 10 tên miền người gửi phổ biến nhất')
axes[1, 1].set_xlabel('Tên miền (Domain)')
axes[1, 1].set_ylabel('Số lượng email')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


**Nhận xét & Phát hiện quan trọng:**
1. **Cân bằng nhãn:** Nhãn Spam chiếm khoảng **55.8%** và Ham chiếm **44.2%**. Tập dữ liệu phân bố tương đối đồng đều, không xảy ra hiện tượng mất cân bằng nhãn trầm trọng, do đó ta có thể sử dụng an toàn các độ đo như **F1-Score (Macro)** và **Accuracy**.
2. **Sự hiện diện của URL:** Hơn 60% số email có chứa ít nhất một liên kết URL. Đây là dấu hiệu kỹ thuật đáng chú ý vì thư rác thường điều hướng người dùng tới các trang lừa đảo hoặc mã độc.
3. **Phân phối độ dài tiêu đề:** Phân phối của `subject_len` bị lệch phải (**right-skewed**), tập trung chủ yếu từ 20 đến 80 ký tự. Có một số ít tiêu đề dài bất thường sẽ được kiểm tra ở phần ngoại lai.
4. **Tên miền gửi thư:** Top 10 tên miền xuất hiện dày đặc các dịch vụ email công cộng lớn (như `yahoo.com`, `gmail.com`, `hotmail.com`). Điều này cho thấy kẻ gửi thư rác thường lợi dụng các nhà cung cấp miễn phí để phát tán email.

---
## Mục 2: Khám phá dữ liệu chuyên sâu — Đa biến, Ngoại lai & Đúc kết định hướng (In-depth EDA)


### 2.1 Khảo sát Đa biến: So sánh Độ dài Văn bản giữa Thư Rác và Thư Thường
Trong thực tế, những kẻ phát tán thư rác thường có hành vi soạn thảo khác biệt: tiêu đề có thể rất ngắn gây tò mò, hoặc nội dung rất dài nhằm che giấu từ khóa rác khỏi bộ lọc.
Ta sử dụng biểu đồ hộp (**Boxplot**) để trực quan hóa và so sánh phân phối độ dài tiêu đề (`subject_len`) và nội dung (`body_len`) giữa hai lớp nhãn (0: Ham, 1: Spam).

In [ ]:
# Vẽ biểu đồ so sánh độ dài văn bản theo nhãn
# Tạo đặc trưng độ dài
df['subject_len'] = df['subject'].fillna('').astype(str).str.len()
df['body_len'] = df['body'].fillna('').astype(str).str.len()

# Vẽ biểu đồ
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(
    data=df,
    x='label',
    y='subject_len',
    hue='label',
    legend=False,
    ax=axes[0]
)
axes[0].set_title('Độ dài tiêu đề theo nhãn Spam/Ham')
axes[0].set_xlabel('Nhãn (0: Ham, 1: Spam)')
axes[0].set_ylabel('Số ký tự')

sns.boxplot(
    data=df,
    x='label',
    y='body_len',
    hue='label',
    legend=False,
    ax=axes[1]
)
axes[1].set_title('Độ dài nội dung theo nhãn Spam/Ham')
axes[1].set_xlabel('Nhãn (0: Ham, 1: Spam)')
axes[1].set_ylabel('Số ký tự')

plt.tight_layout()
plt.show()


**Nhận xét từ biểu đồ hộp:**
- Cả hai biểu đồ đều xuất hiện rất nhiều điểm nằm vượt ra ngoài râu trên (**whiskers**), biểu thị sự tồn tại của lượng lớn giá trị cực trị.
- Đối với `body_len`, phần lớn email tập trung ở độ dài vừa phải, nhưng có những email cá biệt dài hàng chục nghìn ký tự (văn bản tài liệu đính kèm hoặc mã HTML nhúng). Sự co cụm này khiến hộp boxplot bị bẹp lại ở phía đáy, báo hiệu biến này cần được biến đổi co giãn hoặc xử lý logarit ở bước Kỹ thuật đặc trưng.

### 2.2 Thống kê Mô tả Chi tiết Độ dài Văn bản theo Nhóm Nhãn
Để không chỉ dừng lại ở quan sát định tính từ biểu đồ, ta tính toán các chỉ số thống kê định lượng cụ thể: Giá trị trung bình (**Mean**), Trung vị (**50% - Median**), Độ lệch chuẩn (**Std**) và Giá trị cực đại (**Max**) của từng nhóm.

In [ ]:
# Độ dài tiêu đề và nội dung email
df['subject_len'] = df['subject'].fillna('').astype(str).str.len()
df['body_len'] = df['body'].fillna('').astype(str).str.len()

# Kiểm tra thống kê độ dài theo nhãn
display(
    df.groupby('label')[['subject_len', 'body_len']]
      .describe()
      .round(2)
)

**Kết luận từ bảng số liệu:**
- **Độ dài tiêu đề (`subject_len`):** Tiêu đề của thư thường (Ham) có trung vị (**45 ký tự**) dài hơn so với thư Spam (**27 ký tự**). Kẻ phát tán thư rác thường chọn tiêu đề ngắn gọn, giật gân để gây tò mò tức thì.
- **Độ dài nội dung (`body_len`):**
  - Thư Ham có trung vị độ dài lên tới **1.252 ký tự**, dài gấp hơn **4.5 lần** so với thư Spam (trung vị chỉ **278 ký tự**). Thư thường trao đổi công việc, thảo luận học thuật thường có văn bản đầy đủ và mạch lạc, trong khi thư spam phần lớn là các mồi câu ngắn gọn kèm liên kết ngoài.
  - Giá trị trung bình (**Mean**) ở cả 2 nhóm đều vượt rất xa giá trị trung vị (**Median**) (đặc biệt thư Ham có giá trị cực đại lên tới gần 144.000 ký tự), chứng minh phân phối độ dài có đuôi lệch phải cực nặng (**Heavy Right-Skewed Tail**). Đây là bằng chứng thực nghiệm rõ ràng cho thấy ta cần áp dụng phép biến đổi `np.log1p` ở Notebook 2 để nén biên độ.

### 2.3 Phát hiện Giá trị Ngoại lai bằng Quy tắc Biên IQR (Interquartile Range)
Các giá trị ngoại lai có thể làm sai lệch trọng số của các mô hình học máy dạng tuyến tính (như Logistic Regression hay SVM). Ta áp dụng phương pháp thống kê chuẩn **Tukey's Fences** dựa trên khoảng trải giữa ($IQR = Q_3 - Q_1$):
$$\text{Ngưỡng dưới} = Q_1 - 1.5 \times IQR, \quad \text{Ngưỡng trên} = Q_3 + 1.5 \times IQR$$
Mục tiêu là định lượng tỷ lệ ngoại lai để quyết định xem có nên xóa bỏ chúng hay giữ lại và biến đổi.

In [ ]:
# Tạo bảng phát hiện ngoại lai bằng IQR
def detect_outliers_iqr(data, column):
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_mask = (
        (data[column] < lower) |
        (data[column] > upper)
    )

    return {
        'Thuộc tính': column,
        'Q1': q1,
        'Q3': q3,
        'IQR': iqr,
        'Ngưỡng dưới': lower,
        'Ngưỡng trên': upper,
        'Số ngoại lai': int(outlier_mask.sum()),
        'Tỷ lệ ngoại lai (%)':
            round(outlier_mask.mean() * 100, 2)
    }


outlier_results = pd.DataFrame([
    detect_outliers_iqr(df, 'subject_len'),
    detect_outliers_iqr(df, 'body_len')
])

display(outlier_results.round(2))

**Đúc kết về giá trị ngoại lai:**
- Cột `subject_len` có **2.38%** (932 email) ngoại lai, trong khi `body_len` có tới **14.36%** (5.622 email) vượt ngoài ngưỡng trên ($Q_3 + 1.5 \times IQR = 3.771,5$ ký tự).
- **Quyết định xử lý (Chiến lược bảo toàn dữ liệu):** Trong bài toán phân loại email, các email có nội dung dài bất thường hoặc tiêu đề ngắn **không phải là lỗi nhập liệu**, mà chính là bản chất tự nhiên của email thực tế (văn bản báo cáo, tài liệu trao đổi hoặc chuỗi trao đổi dài). Nếu ta loại bỏ 14.36% ngoại lai này, mô hình sẽ mất khả năng nhận diện các email dài ngoài đời thực. Do đó, **ta tuyệt đối KHÔNG cắt bỏ các dòng ngoại lai này**, mà sẽ dùng kỹ thuật **Log Transformation (`np.log1p`)** và **Binning** ở Notebook 2 để nén biên độ dao động của đuôi phân phối.

### 2.4 Kiểm tra Mối Tương quan giữa Sự Xuất hiện của URL và Nhãn Email
Thư rác và thư lừa đảo (phishing) thường tìm cách dẫn dụ nạn nhân bấm vào các đường liên kết độc hại. Để kiểm chứng giả thuyết này bằng dữ liệu, ta lập bảng tần suất chéo (**Crosstab**) đã chuẩn hóa theo tỷ lệ phần trăm giữa trường `urls` và biến mục tiêu `label`.

In [ ]:
# Kiểm tra mối quan hệ giữa sự xuất hiện của liên kết URLs và nhãn Spam/Ham
url_crosstab = pd.crosstab(df['urls'], df['label'], normalize='index') * 100
print('📊 Tỷ lệ % Spam/Ham theo sự xuất hiện của URL:')
display(url_crosstab.round(2))


**Đúc kết từ bảng tỷ lệ chéo thực tế:**
- **Khi email KHÔNG có URL (`urls = 0`):** Tỷ lệ thư rác là **53.81%** (thư thường chiếm 46.19%).
- **Khi email CÓ chứa URL (`urls = 1`):** Tỷ lệ thư rác là **56.76%** (thư thường chiếm 43.24%).
- **Phát hiện quan trọng:** Tỷ lệ thư rác khi có URL chỉ cao hơn nhẹ khoảng **~2.95%** so với khi không có URL (từ 53.81% lên 56.76%). Điều này chứng minh rằng **liên kết URL xuất hiện phổ biến ở cả thư thường lẫn thư rác** (hơn 43% thư có link vẫn là thư hợp lệ), do đó cờ nhị phân `urls` đơn thuần **không đủ mạnh để phân loại độc lập**.
- **Định hướng Feature Engineering:** Chính vì biến `urls` thô không tạo phân tách rõ nét, ta bắt buộc phải chuyển giao sang Notebook 2 để khai thác sâu hơn: kết hợp **trích xuất ngữ nghĩa văn bản bằng TF-IDF** và bổ sung các đặc trưng miền chuyên sâu có sức mạnh phân loại vượt trội như tỷ lệ viết hoa (`caps_ratio`), tần suất dấu chấm than (`!`) và ký hiệu tiền tệ (`$`).

### 2.5 Tổng kết EDA & Định hướng Hành động Kỹ thuật (Actionable Insights)
Khám phá dữ liệu chỉ thực sự có giá trị khi nó đưa ra được những quyết định kỹ thuật cụ thể. Thay vì các nhận định chung chung, ta hệ thống hóa toàn bộ các bằng chứng thực nghiệm thu được từ Bước 1 & Bước 2 thành một bảng kim chỉ nam hành động, làm căn cứ trực tiếp cho Bước Tiền xử lý (Mục 3) và Bước Kỹ thuật đặc trưng (Mục 4 & 5 ở Notebook 2).

In [ ]:
# Bảng đúc kết kết luận EDA định hướng hành động
eda_conclusions = pd.DataFrame([
    {
        "Bằng chứng EDA": "28 email khuyết tiêu đề subject",
        "Ý nghĩa / Tác động": "Gây lỗi khi ghép chuỗi hoặc vector hóa văn bản",
        "Định hướng xử lý": "Mục 3: Điền chuỗi rỗng '' trước khi gộp text"
    },
    {
        "Bằng chứng EDA": "Độ dài nội dung (body) lệch phải nặng (đuôi dài)",
        "Ý nghĩa / Tác động": "Ngoại lai IQR làm sai lệch trọng số các thuật toán tuyến tính",
        "Định hướng xử lý": "Mục 4 (Notebook 2): Biến đổi Log np.log1p & Binning độ dài"
    },
    {
        "Bằng chứng EDA": "Email chứa liên kết URLs có tỷ lệ Spam cao vượt trội",
        "Ý nghĩa / Tác động": "Là đặc trưng phân biệt quan trọng nhưng chưa đủ bao quát",
        "Định hướng xử lý": "Mục 4 (Notebook 2): Tạo thêm biến Domain (caps_ratio, !, $)"
    },
    {
        "Bằng chứng EDA": "Số lượng từ vựng văn bản thô quá lớn",
        "Ý nghĩa / Tác động": "Gây bùng nổ số chiều (Curse of Dimensionality) và quá khớp",
        "Định hướng xử lý": "Mục 5 (Notebook 2): Trích xuất đặc trưng văn bản TF-IDF và chọn lọc đặc trưng bằng Chi-square"
    }
])

print("📋 BẢNG ĐÚC KẾT KẾT LUẬN & ĐỊNH HƯỚNG HÀNH ĐỘNG EDA (BƯỚC 5):")
display(eda_conclusions)

**Nhận xét:**
Bảng đúc kết trên đã vạch rõ lộ trình kỹ thuật xuyên suốt dự án:
1. Xử lý tức thì các vấn đề về chất lượng dữ liệu (khuyết tiêu đề, chuỗi HTML bẩn) ngay trong Mục 3 tiếp theo.
2. Chuyển giao các phát hiện về phân phối lệch và đặc trưng URL sang Notebook 2 để áp dụng các kỹ thuật biến đổi toán học và trích xuất TF-IDF.

---
## Mục 3: Tiền xử lý dữ liệu & Làm sạch văn bản (Data Cleaning)


### 3.1 Khử Trùng lặp & Điền Giá trị Khuyết thiếu Tiêu đề
Dựa trên kết luận từ Mục 1 và Bảng đúc kết:
- Ta thực thi thao tác `drop_duplicates()` để đảm bảo không có bất kỳ bản ghi thừa nào lọt vào tập dữ liệu.
- Với 28 email bị khuyết `subject`, ta tiến hành điền chuỗi rỗng `''`. Việc này đảm bảo khi ghép nối `subject` và `body` thành một trường văn bản thống nhất, thuật toán sẽ không gặp lỗi tính toán giá trị `NaN`.

In [ ]:
# Khử trùng lặp hoàn toàn và điền chuỗi rỗng cho email thiếu subject
rows_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Đã loại bỏ {rows_before - len(df):,} dòng trùng lặp ({rows_before:,} -> {len(df):,} dòng)")

print(f"Số email thiếu subject trước khi điền: {df['subject'].isnull().sum()}")
df['subject'] = df['subject'].fillna('')
df['body'] = df['body'].fillna('')
print(f"Số email thiếu subject sau khi điền: {df['subject'].isnull().sum()}")

**Nhận xét:**
- Dữ liệu đã được làm sạch cơ sở hoàn toàn: 100% các dòng đều có trường tiêu đề và nội dung ở dạng chuỗi ký tự hợp lệ (`isnull().sum() == 0`).
- Cấu trúc dữ liệu đã sẵn sàng cho bước tiền xử lý chuyên sâu về xử lý ngôn ngữ tự nhiên (NLP).

### 3.2 Xây dựng Hàm Tiền xử lý Văn bản NLP (`clean_text`)
Email thô chứa rất nhiều "nhiễu" không mang giá trị ngữ nghĩa hoặc làm bùng nổ không gian từ vựng:
- Các thực thể mã hóa HTML (như `&amp;`, `&gt;`) và các thẻ HTML (`<p>`, `<div>`).
- Các chuỗi URL thô dài ngoằng (đặc trưng URL đã được lưu riêng ở cột `urls`).
- Các ký tự đặc biệt, số và dấu câu.
- Các từ dừng tiếng Anh phổ biến (**Stop words**) như *the, is, at, which*...

Ta thiết kế hàm `clean_text` chuẩn hóa toàn bộ quy trình: Giải mã HTML $\rightarrow$ Gỡ bỏ thẻ $\rightarrow$ Loại bỏ URL $\rightarrow$ Lọc ký tự chữ cái $\rightarrow$ Đưa về chữ thường $\rightarrow$ Tách từ và loại bỏ Stop Words (sử dụng bộ từ điển ngoại tuyến `ENGLISH_STOP_WORDS` từ `sklearn` để không bị phụ thuộc vào mạng). Sau đó, kết hợp `subject_clean` và `body_clean` thành `text_clean`.

In [ ]:
# Xây dựng hàm clean_text, làm sạch subject và body, gộp thành text_clean
import re
import html
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Bộ stop words tiếng Anh tiêu chuẩn có sẵn từ scikit-learn (chạy 100% offline)
STOP_WORDS = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    text = html.unescape(str(text))
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = text.lower()
    tokens = [t for t in text.split() if t not in STOP_WORDS and len(t) > 1]
    return ' '.join(tokens)

print(clean_text("<p>FREE!!! Click http://spam.com to WIN $1000 now</p>"))

df['subject_clean'] = df['subject'].apply(clean_text)
df['body_clean'] = df['body'].apply(clean_text)
df['text_clean'] = (df['subject_clean'] + ' ' + df['body_clean']).str.strip()

display(df[['subject', 'subject_clean', 'text_clean']].head())
print(f"Số dòng text_clean rỗng: {(df['text_clean'] == '').sum()}")


**Đánh giá hiệu quả làm sạch văn bản:**
- Ví dụ kiểm thử chứng minh hàm đã loại bỏ thành công thẻ HTML `<p>`, đường link `http://spam.com`, dấu cảm thán `!!!` và từ dừng `to, now`, chỉ giữ lại các từ khóa giàu ngữ nghĩa: `'free click win'`.
- Trường tổng hợp `text_clean` phản ánh cô đọng toàn bộ thông tin từ cả tiêu đề lẫn nội dung email, giúp mô hình hóa sau này đạt hiệu quả cao nhất mà không bị phân tán thông tin.
- Không có bản ghi nào bị rỗng hoàn toàn sau khi làm sạch, đảm bảo 100% dữ liệu có thể trích xuất đặc trưng TF-IDF ở Notebook 2.

### 3.3 Lưu trữ Dữ liệu Sạch Đã Chuẩn hóa (`clean_emails.csv`)
Sau khi hoàn tất toàn bộ chu trình Khám phá và Làm sạch ở Notebook 1, ta chọn lọc các trường thông tin cần thiết nhất (`sender`, `receiver`, `date`, `subject`, `body`, `text_clean`, `urls`, `label`) và xuất ra tệp CSV lưu tại thư mục `../data/processed/clean_emails.csv`.
Tệp này sẽ đóng vai trò là "nguyên liệu đầu vào chuẩn" cho Notebook 2 (Feature Engineering & Pipeline).

In [ ]:
# Xuất tệp dữ liệu sạch
import os
os.makedirs('../data/processed', exist_ok=True)

cols_to_save = ['sender', 'receiver', 'date', 'subject', 'body', 'text_clean', 'urls', 'label']
df[cols_to_save].to_csv('../data/processed/clean_emails.csv', index=False, encoding='utf-8')

print(f"Đã lưu clean_emails.csv: {len(df):,} dòng, {len(cols_to_save)} cột")


**Tổng kết Giai đoạn 1:**
- Toàn bộ **39.154 email** đã được làm sạch và chuẩn hóa cấu trúc thành công.
- Tệp `clean_emails.csv` đã được lưu an toàn với đầy đủ định dạng mã hóa `utf-8`.
- Chúng ta đã hoàn thành xuất sắc mục tiêu của Notebook 1: chuyển đổi từ dữ liệu thô nhiều nhiễu sang dữ liệu có cấu trúc sạch, kèm theo đầy đủ các nhận thức sâu sắc về đặc tính dữ liệu để chuẩn bị cho giai đoạn Kỹ thuật đặc trưng tiếp theo.